# GNSS-Denied Navigation — Satellite-to-Drone Cross-View Matching

Train a feature extractor to match drone downward camera views against satellite/aerial reference imagery.
Enables autonomous position estimation without GPS.

**Approach:** Contrastive learning (triplet loss) — learn embeddings where same-location drone+satellite pairs are close, different locations are far apart.

**Dataset:** University-1652 (drone ↔ satellite cross-view pairs) + custom ESRI tiles

**Result:** Feature extractor that can localize a drone image against a pre-loaded satellite map.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/DroneCV/gnss_denied'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

In [ ]:
!pip install -q torch torchvision timm gdown

## Step 1: Download University-1652 Dataset

Cross-view geo-localization dataset with drone, satellite, and ground views of 1,652 university buildings.
- Drone views: ~37k images from various altitudes/angles
- Satellite views: 1,652 images (one per location)
- Perfect for training drone→satellite matching

In [ ]:
# Download University-1652 dataset
# Source: https://github.com/layumi/University1652-Baseline
!gdown 1iVnP4gjw-iHXa0KerZQ1IfIO02aOPz3y -O /content/University-1652.zip
!unzip -q /content/University-1652.zip -d /content/data/

In [ ]:
import glob

# Check dataset structure
train_drone = sorted(glob.glob('/content/data/University-1652/train/drone/*/*.jpg'))
train_sat = sorted(glob.glob('/content/data/University-1652/train/satellite/*/*.jpg'))
print(f'Training: {len(train_drone)} drone images, {len(train_sat)} satellite images')
print(f'Locations: {len(set(os.path.basename(os.path.dirname(p)) for p in train_drone))}')

## Step 2: Define Cross-View Feature Extractor

Use a pretrained backbone (ResNet50 or EfficientNet) with a shared embedding head.
Both drone and satellite images are mapped to the same 512-d embedding space.

In [ ]:
import torch
import torch.nn as nn
import timm

class CrossViewEncoder(nn.Module):
    """Shared encoder for drone and satellite views."""
    def __init__(self, backbone='resnet50', embed_dim=512):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.embed = nn.Sequential(
            nn.Linear(feat_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embed(features)
        return nn.functional.normalize(embedding, p=2, dim=1)

model = CrossViewEncoder('resnet50', embed_dim=512).cuda()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 3: Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

class CrossViewDataset(Dataset):
    """Triplet dataset: (drone, satellite_pos, satellite_neg)."""
    def __init__(self, root, split='train', img_size=256):
        self.root = os.path.join(root, split)
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        # Build location→images mapping
        self.drone_imgs = {}
        self.sat_imgs = {}
        for p in glob.glob(os.path.join(self.root, 'drone', '*', '*.jpg')):
            loc = os.path.basename(os.path.dirname(p))
            self.drone_imgs.setdefault(loc, []).append(p)
        for p in glob.glob(os.path.join(self.root, 'satellite', '*', '*.jpg')):
            loc = os.path.basename(os.path.dirname(p))
            self.sat_imgs.setdefault(loc, []).append(p)
        self.locations = sorted(set(self.drone_imgs) & set(self.sat_imgs))
        print(f'[{split}] {len(self.locations)} locations with both drone+satellite')

    def __len__(self):
        return len(self.locations) * 5  # 5 triplets per location per epoch

    def __getitem__(self, idx):
        loc = self.locations[idx % len(self.locations)]
        # Positive pair: drone + satellite from same location
        drone_img = Image.open(random.choice(self.drone_imgs[loc])).convert('RGB')
        sat_pos = Image.open(random.choice(self.sat_imgs[loc])).convert('RGB')
        # Negative: satellite from different location
        neg_loc = random.choice([l for l in self.locations if l != loc])
        sat_neg = Image.open(random.choice(self.sat_imgs[neg_loc])).convert('RGB')
        return self.transform(drone_img), self.transform(sat_pos), self.transform(sat_neg)

train_ds = CrossViewDataset('/content/data/University-1652', 'train')
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

## Step 4: Training (Triplet Loss)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
triplet_loss = nn.TripletMarginLoss(margin=0.3)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, (drone, sat_pos, sat_neg) in enumerate(train_loader):
        drone, sat_pos, sat_neg = drone.cuda(), sat_pos.cuda(), sat_neg.cuda()

        # Forward
        emb_drone = model(drone)
        emb_pos = model(sat_pos)
        emb_neg = model(sat_neg)

        loss = triplet_loss(emb_drone, emb_pos, emb_neg)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}')

# Save model
torch.save(model.state_dict(), f'{WORK_DIR}/models/crossview_resnet50_512d.pth')
print(f'Model saved to {WORK_DIR}/models/crossview_resnet50_512d.pth')

## Step 5: Evaluate — Retrieval Accuracy

Given a drone image, find the correct satellite tile (top-1, top-5 accuracy).

In [ ]:
import numpy as np

model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Build satellite gallery embeddings
test_sat = sorted(glob.glob('/content/data/University-1652/test/gallery_satellite/*/*.jpg'))
test_drone = sorted(glob.glob('/content/data/University-1652/test/query_drone/*/*.jpg'))

def embed_images(paths, model, transform, batch_size=64):
    embeddings, labels = [], []
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        imgs = torch.stack([transform(Image.open(p).convert('RGB')) for p in batch_paths]).cuda()
        with torch.no_grad():
            emb = model(imgs)
        embeddings.append(emb.cpu().numpy())
        labels.extend([os.path.basename(os.path.dirname(p)) for p in batch_paths])
    return np.vstack(embeddings), labels

print('Embedding satellite gallery...')
sat_emb, sat_labels = embed_images(test_sat, model, eval_transform)
print(f'Gallery: {sat_emb.shape}')

print('Embedding drone queries...')
drone_emb, drone_labels = embed_images(test_drone[:500], model, eval_transform)
print(f'Queries: {drone_emb.shape}')

# Compute retrieval accuracy
distances = np.dot(drone_emb, sat_emb.T)  # cosine similarity (embeddings are normalized)
top1, top5, top10 = 0, 0, 0
for i, query_label in enumerate(drone_labels):
    ranked = np.argsort(-distances[i])
    retrieved_labels = [sat_labels[r] for r in ranked]
    if query_label in retrieved_labels[:1]: top1 += 1
    if query_label in retrieved_labels[:5]: top5 += 1
    if query_label in retrieved_labels[:10]: top10 += 1

n = len(drone_labels)
print(f'\nRetrieval Accuracy:')
print(f'  Top-1:  {top1/n*100:.1f}%')
print(f'  Top-5:  {top5/n*100:.1f}%')
print(f'  Top-10: {top10/n*100:.1f}%')

## Step 6: Test on Custom Satellite Tiles

Load our own ESRI tiles and test matching against a simulated drone view.

In [ ]:
# Upload your satellite tiles to Drive, or clone from repo:
# !git clone https://github.com/rwiren/drone-cv-detection.git /content/repo
# Then copy: data/satellite_tiles/jorvas_satellite_z18_stitched.jpg

import cv2

# For demo: create tiles from stitched reference
ref_path = f'{WORK_DIR}/data/jorvas_satellite_z18_stitched.jpg'
if os.path.exists(ref_path):
    ref = cv2.imread(ref_path)
    # Split into 3x3 grid of 256x256 tiles
    tile_embeddings = []
    tile_centers = []
    tile_size = 256
    for row in range(3):
        for col in range(3):
            tile = ref[row*tile_size:(row+1)*tile_size, col*tile_size:(col+1)*tile_size]
            tile_rgb = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)
            tile_tensor = eval_transform(Image.fromarray(tile_rgb)).unsqueeze(0).cuda()
            with torch.no_grad():
                emb = model(tile_tensor)
            tile_embeddings.append(emb.cpu().numpy().flatten())
            tile_centers.append((col * tile_size + tile_size//2, row * tile_size + tile_size//2))

    tile_embeddings = np.array(tile_embeddings)
    print(f'Embedded {len(tile_embeddings)} reference tiles')

    # Simulate drone view (crop from reference + augment)
    drone_crop = ref[200:456, 200:456]  # 256x256 crop
    drone_crop = cv2.GaussianBlur(drone_crop, (3,3), 0)
    drone_rgb = cv2.cvtColor(drone_crop, cv2.COLOR_BGR2RGB)
    drone_tensor = eval_transform(Image.fromarray(drone_rgb)).unsqueeze(0).cuda()
    with torch.no_grad():
        drone_emb = model(drone_tensor).cpu().numpy().flatten()

    # Find best matching tile
    similarities = np.dot(tile_embeddings, drone_emb)
    best_idx = np.argmax(similarities)
    print(f'Best match: tile {best_idx}, center={tile_centers[best_idx]}, similarity={similarities[best_idx]:.3f}')
    print(f'True position: (328, 328) — error: {np.sqrt((tile_centers[best_idx][0]-328)**2 + (tile_centers[best_idx][1]-328)**2) * 0.30:.1f}m')
else:
    print(f'Upload {ref_path} to test on custom tiles')

## Summary

**What we trained:** A ResNet50-based cross-view feature extractor (512-d embeddings).

**How it's used for GNSS-denied flight:**
1. Pre-flight: embed all satellite tiles of the flight area → build gallery
2. In-flight: embed downward camera frame → find nearest satellite tile → get position
3. Send position to ArduPilot via `VISION_POSITION_ESTIMATE`

**Integration:** The trained model runs in `gnss_denied_nav.py` on the ground server,
receiving the drone's downward camera RTSP stream over 5G.

**Next steps:**
- Fine-tune on Finnish NLS orthophotos for local terrain
- Add rotation invariance (drone heading unknown without GPS)
- Test at multiple altitudes (10m, 30m, 50m, 100m)